# nb41 - H6: Clean-teacher feature distillation at 9x9

**Error analysis.** Per-cell fraction labels from overlays failed as LABELS: nb28 multitask supervision left the energy head unmoved (0.0502 -> 0.0512) and nb30's load-bearing wiring plus DANN could not close the overlay-vs-real domain gap (closure AUC 0.74). But the PAIRED construction itself is exact: for every overlay we hold both the clean window A and the contaminated window A+pileup of the same photon.

**Question.** Can the exact pairing be exploited without asking the network to trust synthetic per-cell labels?

**Hypothesis H6.** Feature-space distillation transfers where labels did not: a frozen teacher that sees the CLEAN view produces an embedding the student must match while seeing the CONTAMINATED view.

**Research.** Hong et al., CVPR 2021 (arXiv:2103.07600) distill a clean-input teacher into a degraded-input student; FitNets (arXiv:1412.6550) established intermediate-feature matching; paired-simulation teacher/student (arXiv:1901.02348) reports a 19.6% relative gain on a real test set.

**Proof judgement.** `distill` vs `nodistill` (lam_f = 0) on IDENTICAL data (real train + merged overlays), same seeds; early stop and calibration on real val, sigma_eff on the real min-bias test split. Differences < 0.002 are not significant.

**Code.** W=4 (9x9), THRESH 2.49 MeV, SubNet d=128 subtract-then-calibrate readout returning (pred, embedding p); teacher trained on clean windows only (seed 0, frozen after training), student loss = Huber(delta 0.1) + lam_f * SmoothL1(p_student_merged, p_teacher_cleanview) applied only to overlay rows; lam_f warms 0 -> 0.5 over the first 5 epochs then cosine-decays to 0.1 by the last epoch. Overlays at W=4: clean recipient, up to 4 real-train donors (rmax >= 10) at Chebyshev offset 6-7, contamination matched to real W=4 window ratios, seed-displacement drop, cap 40000. Anchors: nb32 W4 0.0459-0.0471 (ens 0.0450), nb35 final 0.0440.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd, uproot, awkward as ak
import torch, torch.nn as nn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS
CLEAN = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
MB = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
OUT = REPO / 'reports' / 'predictions'; OUT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB41_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB41_MODE', 'full')
if MODE == 'smoke': CLEAN, MB = CLEAN[:4], MB[:8]
THRESH = 2.49
W = 4; L = (2*W+1)**2
print('device', DEVICE, '| mode', MODE, '|', len(CLEAN), 'clean,', len(MB), 'minbias | W', W, '| THRESH', THRESH, 'MeV')

device cpu | mode full | 100 clean, 94 minbias | W 4 | THRESH 2.49 MeV


In [2]:
TK = ['cell_x','cell_y','energy','cell_energies_front','cell_energies_back',
      'cell_times_front','cell_times_back','imodx','jmody']
AUX = ['sig_flux_prod_vertex_z','sig_flux_eTot']
def event_geom(cc):
    x, yy, e = cc['cell_x'], cc['cell_y'], cc['energy']
    ix, iy = cc['imodx'], cc['jmody']
    seed = int(np.argmax(e))
    pts = np.stack([x, yy], 1)
    pitch = np.full(len(x), np.nan)
    for key in {(int(p), int(q)) for p, q in zip(ix, iy)}:
        sel = (ix == key[0]) & (iy == key[1]); p = pts[sel]
        if len(p) >= 2:
            d = np.sqrt(((p[:, None, :] - p[None, :, :]) ** 2).sum(-1)); d[d == 0] = np.inf
            pitch[sel] = np.median(np.min(d, axis=1))
    fill = np.nanmedian(pitch) if np.isfinite(pitch).any() else 120.0
    pitch[~np.isfinite(pitch)] = fill
    ps = pitch[seed]
    ei = (x - x[seed]) / ps; ej = (yy - yy[seed]) / ps
    di = np.round(ei).astype(int); dj = np.round(ej).astype(int)
    ok = (np.abs(ei - di) < 0.15) & (np.abs(ej - dj) < 0.15)
    return seed, ps, di, dj, ok
def build_grid(files, keep_cheby, label):
    EV = []
    for path in files:
        with uproot.open(path) as f:
            a = f['clusters_matched'].arrays(TK + AUX, library='ak')
        vz = ak.to_numpy(a['sig_flux_prod_vertex_z']).astype(float)
        et_all = ak.to_numpy(a['sig_flux_eTot']).astype(float)
        for i in np.flatnonzero((vz < 100.0) & (et_all >= 1.0) & (et_all <= 100.0)):
            cc = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in TK}
            e = cc['energy']
            if len(e) < 3: continue
            seed, ps, di, dj, ok = event_geom(cc)
            if ok.mean() < 0.5: continue
            ch = np.maximum(np.abs(di), np.abs(dj))
            keep = ok if keep_cheby is None else (ok & (ch <= keep_cheby))
            if keep.sum() < 1 or not keep[seed]: continue
            tf = cc['cell_times_front'][keep]; tb = cc['cell_times_back'][keep]
            tf = np.where(np.isfinite(tf) & (tf != 0) & (np.abs(tf) < 1e4), tf, np.nan)
            tb = np.where(np.isfinite(tb) & (tb != 0) & (np.abs(tb) < 1e4), tb, np.nan)
            EV.append(dict(di=di[keep].astype(np.int16), dj=dj[keep].astype(np.int16),
                           e=e[keep].astype(np.float32),
                           fr=cc['cell_energies_front'][keep].astype(np.float32),
                           bk=cc['cell_energies_back'][keep].astype(np.float32),
                           tf=tf.astype(np.float32), tb=tb.astype(np.float32),
                           ps=float(ps), reg=int(np.argmin(np.abs(PITCH - ps))),
                           rmax=int(ch[ok].max()), Etrue=float(et_all[i])))
    print(f'{label}: {len(EV)} events')
    return EV
t0 = time.time()
CE = build_grid(CLEAN, 4, 'clean')
ME = build_grid(MB, None, 'minbias')
print(f'build {time.time()-t0:.0f}s')

clean: 30303 events


minbias: 72554 events
build 129s


In [3]:
def window_tokens(di, dj, e, fr, bk, tf, tb, ps, reg):
    m = (np.maximum(np.abs(di), np.abs(dj)) <= W) & (e >= THRESH)
    if m.sum() < 1: return None
    di, dj, e, fr, bk, tf, tb = (v[m] for v in (di, dj, e, fr, bk, tf, tb))
    t0f = np.nanmedian(tf) if np.isfinite(tf).any() else 0.0
    t0b = np.nanmedian(tb) if np.isfinite(tb).any() else 0.0
    tfc = np.where(np.isfinite(tf), tf - t0f, 0.0); htf = np.isfinite(tf).astype(np.float32)
    tbc = np.where(np.isfinite(tb), tb - t0b, 0.0); htb = np.isfinite(tb).astype(np.float32)
    rdr = np.hypot(di, dj)
    cont = np.stack([np.log1p(np.clip(e, 0, None)), np.log1p(np.clip(fr, 0, None)),
                     np.log1p(np.clip(bk, 0, None)), di.astype(np.float32), dj.astype(np.float32),
                     rdr, np.full(len(e), np.log(ps)), np.clip(tfc, -5, 5), np.clip(tbc, -5, 5)], 1)
    oh = np.zeros((len(e), len(PITCH)), np.float32); oh[:, reg] = 1.0
    tok = np.concatenate([cont, htf[:, None], htb[:, None], oh], 1).astype(np.float32)
    return tok, float(e.sum()), float(e.max())
def real_window(ev):
    r = window_tokens(ev['di'], ev['dj'], ev['e'], ev['fr'], ev['bk'], ev['tf'], ev['tb'], ev['ps'], ev['reg'])
    return None if r is None else (r[0], r[1], r[2], ev['Etrue'])
def add_donor(pos, B, rng):
    di0 = dj0 = 0
    while max(abs(di0), abs(dj0)) not in (6, 7):
        di0 = int(rng.integers(-7, 8)); dj0 = int(rng.integers(-7, 8))
    for k in range(len(B['e'])):
        ri, rj = int(B['di'][k]) - di0, int(B['dj'][k]) - dj0
        if max(abs(ri), abs(rj)) > W: continue
        eb = float(B['e'][k]); tfb = float(B['tf'][k]); tbb = float(B['tb'][k])
        if (ri, rj) in pos:
            p = pos[(ri, rj)]
            for idx, val in ((3, tfb), (4, tbb)):
                if np.isfinite(val) and np.isfinite(p[idx]):
                    p[idx] = (p[idx] * p[0] + val * eb) / (p[0] + eb)
                elif np.isfinite(val): p[idx] = val
            p[0] += eb; p[1] += float(B['fr'][k]); p[2] += float(B['bk'][k])
        else:
            pos[(ri, rj)] = [eb, float(B['fr'][k]), float(B['bk'][k]), tfb, tbb]
def overlay(A, pool, rng, target_ratio):
    pos = {}
    for k in range(len(A['e'])):
        if max(abs(A['di'][k]), abs(A['dj'][k])) <= W:
            pos[(int(A['di'][k]), int(A['dj'][k]))] = [float(A['e'][k]), float(A['fr'][k]), float(A['bk'][k]),
                                                       float(A['tf'][k]), float(A['tb'][k])]
    if not pos: return None
    et_mev = A['Etrue'] * 1e3
    nd = 0
    while nd < 4:
        if sum(p[0] for p in pos.values()) / et_mev >= target_ratio: break
        add_donor(pos, pool[int(rng.integers(len(pool)))], rng); nd += 1
    keys = list(pos.keys()); V = np.array([pos[k] for k in keys], np.float64)
    di = np.array([k[0] for k in keys], np.int16); dj = np.array([k[1] for k in keys], np.int16)
    if not (di[np.argmax(V[:, 0])] == 0 and dj[np.argmax(V[:, 0])] == 0): return None
    r = window_tokens(di, dj, V[:, 0], V[:, 1], V[:, 2], V[:, 3], V[:, 4], A['ps'], A['reg'])
    return None if r is None else (r[0], r[1], r[2], A['Etrue'])

In [4]:
n_real = len(ME)
rtr0, rva0, rte0 = split(n_real)
REAL = []; keep_real = []
for i, ev in enumerate(ME):
    r = real_window(ev)
    if r is not None: REAL.append(r); keep_real.append(i)
keep_real = np.array(keep_real)
remap = -np.ones(n_real, int); remap[keep_real] = np.arange(len(REAL))
rtr = remap[rtr0][remap[rtr0] >= 0]; rva = remap[rva0][remap[rva0] >= 0]; rte = remap[rte0][remap[rte0] >= 0]
ratio_pool = np.array([REAL[i][1] / (REAL[i][3] * 1e3) for i in rtr])
donors_by_reg = {}
for i in rtr0:
    ev = ME[i]
    if ev['rmax'] >= 10: donors_by_reg.setdefault(ev['reg'], []).append(ev)
CW = []; keep_cw = []
for i, ev in enumerate(CE):
    r = real_window(ev)
    if r is not None: CW.append(r); keep_cw.append(i)
cw_remap = -np.ones(len(CE), int); cw_remap[np.array(keep_cw)] = np.arange(len(CW))
ttr, tva, tte = split(len(CW))
rng = np.random.default_rng(0)
MAX_OVR = 1500 if MODE == 'smoke' else 40000
OVR = []; a_rows = []; dropped = 0
for idx in rng.permutation(len(CE)):
    if len(OVR) >= MAX_OVR: break
    if cw_remap[idx] < 0: continue
    A = CE[idx]
    pool = donors_by_reg.get(A['reg'])
    if not pool: continue
    tgt = float(ratio_pool[int(rng.integers(len(ratio_pool)))])
    r = overlay(A, pool, rng, tgt)
    if r is None: dropped += 1; continue
    OVR.append(r); a_rows.append(int(cw_remap[idx]))
a_rows = np.array(a_rows, int)
print(f'real {len(REAL)} (tr/va/te {len(rtr)}/{len(rva)}/{len(rte)}) | clean windows {len(CW)} '
      f'(tr/va/te {len(ttr)}/{len(tva)}/{len(tte)}) | donors {sum(len(v) for v in donors_by_reg.values())} '
      f'| overlays {len(OVR)} (dropped {dropped})')

real 72554 (tr/va/te 50787/10883/10884) | clean windows 30303 (tr/va/te 21212/4545/4546) | donors 6135 | overlays 8780 (dropped 937)


In [5]:
ALL = REAL + OVR
N = len(ALL); IN_DIM = REAL[0][0].shape[1]; NG = 5; NC = 9
otr = np.arange(len(REAL), N)
def pack(rows):
    n = len(rows)
    X = np.zeros((n, L, IN_DIM), np.float32); M = np.zeros((n, L), np.bool_)
    G = np.zeros((n, NG), np.float32); Er = np.zeros((n, L), np.float32)
    y = np.zeros(n, np.float32); Et = np.zeros(n, np.float32); sE = np.zeros(n, np.float32)
    for i, (tok, se, sde, et) in enumerate(rows):
        k = tok.shape[0]; X[i, :k] = tok; M[i, :k] = True
        e = np.expm1(tok[:, 0]); Er[i, :k] = e
        lat = float(np.sqrt((e * tok[:, 5] ** 2).sum() / (e.sum() + EPS)))
        fbr = float(np.expm1(tok[:, 1]).sum() / (np.expm1(tok[:, 2]).sum() + EPS))
        G[i] = [np.log1p(se), np.log1p(sde), np.log(k), fbr, lat]
        y[i] = np.log(max(et, 1e-3)); Et[i] = et; sE[i] = se
    return X, M, G, Er, y, Et, sE
Xs, Ms, Gs, Es, ys, Ets, sEs = pack(ALL)
Xt, Mt, Gt, Etc, yt, Ett, sEt = pack(CW)
la0, lb0 = np.polyfit(np.log1p(0.5 * sEs[rtr]), ys[rtr], 1)
la0t, lb0t = np.polyfit(np.log1p(0.5 * sEt[ttr]), yt[ttr], 1)
Gmu = Gs[rtr].mean(0); Gsd = Gs[rtr].std(0) + EPS
Gs = (Gs - Gmu) / Gsd; Gt = (Gt - Gmu) / Gsd
cont = Xs[rtr][:, :, :NC].reshape(-1, NC)[Ms[rtr].reshape(-1)]
mean = cont.mean(0); std = cont.std(0) + EPS
Xs[:, :, :NC] = (Xs[:, :, :NC] - mean) / std; Xs[~Ms] = 0.0
Xt[:, :, :NC] = (Xt[:, :, :NC] - mean) / std; Xt[~Mt] = 0.0
cv = -np.ones(N, np.int64); cv[len(REAL):] = a_rows
T = dict(X=torch.from_numpy(Xs).to(DEVICE), M=torch.from_numpy(Ms).to(DEVICE),
         G=torch.from_numpy(Gs).to(DEVICE), Y=torch.from_numpy(ys).unsqueeze(1).to(DEVICE),
         E=torch.from_numpy(Es).to(DEVICE))
TT = dict(X=torch.from_numpy(Xt).to(DEVICE), M=torch.from_numpy(Mt).to(DEVICE),
          G=torch.from_numpy(Gt).to(DEVICE), Y=torch.from_numpy(yt).unsqueeze(1).to(DEVICE),
          E=torch.from_numpy(Etc).to(DEVICE))
CVt = torch.from_numpy(cv).to(DEVICE)
print(f'student N {N} (real {len(REAL)} + overlay {len(OVR)}), teacher N {len(CW)}, IN_DIM {IN_DIM}, on {DEVICE}')

student N 81334 (real 72554 + overlay 8780), teacher N 30303, IN_DIM 16, on cpu


In [6]:
CFG = dict(d=128, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=96, huber_delta=0.1)
class SubNet(nn.Module):
    def __init__(self, in_dim, la0, lb0):
        super().__init__()
        d = CFG['d']
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                           dropout=CFG['dropout'], batch_first=True)
        self.enc = nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + NG, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 1))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        fl = self.fhead(h).squeeze(-1)
        w = torch.sigmoid(fl) * m.float()
        base = self.la * torch.log1p((w * ecell).sum(1, keepdim=True)) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1)), p
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)

In [7]:
TEP = {'smoke': 2, 'full': 60}[MODE]
TPAT = {'smoke': 99, 'full': 10}[MODE]
def train_teacher():
    torch.manual_seed(0); rng = np.random.default_rng(0)
    model = SubNet(IN_DIM, la0t, lb0t).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TEP)
    ck = CKPT / 'nb41_teacher.pt'
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(b): return model(TT['X'][b], TT['M'][b], TT['G'][b], TT['E'][b])[0]
    def run(idx):
        model.eval(); out = []
        with torch.no_grad():
            for b in batches(idx, 256, False): out.append(fwd(b).cpu().numpy().ravel())
        return np.concatenate(out)
    def vloss():
        model.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(tva, 256, False):
                s += nn.functional.huber_loss(fwd(b), TT['Y'][b], delta=CFG['huber_delta']).item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(1000 * ep0)
        print(f'  resume teacher from epoch {ep0}', flush=True)
    for ep in range(ep0, TEP):
        model.train()
        for b in batches(ttr, CFG['batch'], True):
            opt.zero_grad()
            nn.functional.huber_loss(fwd(b), TT['Y'][b], delta=CFG['huber_delta']).backward()
            opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= TPAT: break
    model.load_state_dict(bstate)
    a, b2 = np.polyfit(run(tva), yt[tva], 1)
    pe = np.exp(a * run(tte) + b2)
    sig = float(resolution(pe, Ett[tte])['sigma_eff'])
    print(f'teacher sigma_eff on clean test: {sig:.4f}', flush=True)
    return model
t0 = time.time()
teacher = train_teacher()
teacher.eval()
for prm in teacher.parameters(): prm.requires_grad_(False)
with torch.no_grad():
    PT = torch.cat([teacher(TT['X'][b], TT['M'][b], TT['G'][b], TT['E'][b])[1]
                    for b in [torch.arange(j, min(j + 256, len(CW)), device=DEVICE)
                              for j in range(0, len(CW), 256)]])
print(f'teacher done ({time.time()-t0:.0f}s), PT {tuple(PT.shape)}')

  resume teacher from epoch 12


teacher sigma_eff on clean test: 0.0809


teacher done (34s), PT (30303, 128)


In [8]:
SEP = {'smoke': 2, 'full': 60}[MODE]
SPAT = {'smoke': 99, 'full': 10}[MODE]
WARM = 5
def lam_at(ep, epochs):
    if ep < WARM: return 0.5 * (ep + 1) / WARM
    t = min((ep - WARM) / max(epochs - 1 - WARM, 1), 1.0)
    return 0.1 + 0.4 * 0.5 * (1.0 + np.cos(np.pi * t))
def train_eval(config, seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNet(IN_DIM, la0, lb0).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ck = CKPT / f'nb41_{config}_s{seed}.pt'
    tr_idx = np.concatenate([np.asarray(rtr), otr])
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def lossf(b, lam):
        pred, p = model(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
        loss = nn.functional.huber_loss(pred, T['Y'][b], delta=CFG['huber_delta'])
        if lam > 0:
            cb = CVt[b]; hb = cb >= 0
            if hb.any():
                loss = loss + lam * nn.functional.smooth_l1_loss(p[hb], PT[cb[hb]].detach())
        return loss
    def run(idx):
        model.eval(); out = []
        with torch.no_grad():
            for b in batches(idx, 256, False):
                out.append(model(T['X'][b], T['M'][b], T['G'][b], T['E'][b])[0].cpu().numpy().ravel())
        return np.concatenate(out)
    def vloss():
        model.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(rva, 256, False):
                pe, _ = model(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
                s += nn.functional.huber_loss(pe, T['Y'][b], delta=CFG['huber_delta']).item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume {config} s{seed} from epoch {ep0}', flush=True)
    for ep in range(ep0, epochs):
        lam = lam_at(ep, epochs) if config == 'distill' else 0.0
        model.train()
        for b in batches(tr_idx, CFG['batch'], True):
            opt.zero_grad(); lossf(b, lam).backward(); opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    model.load_state_dict(bstate)
    a, b2 = np.polyfit(run(rva), ys[rva], 1)
    pe = np.exp(a * run(rte) + b2)
    return float(resolution(pe, Ets[rte])['sigma_eff']), pe

In [9]:
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
CONFIGS = ['distill', 'nodistill']
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb41_distill{TAG}.csv'
done = set()
if CSVP.exists():
    prev = pd.read_csv(CSVP); done = set(zip(prev['config'], prev['seed']))
    print('resume, done:', sorted(done))
for config in CONFIGS:
    for seed in SEEDS:
        if (config, seed) in done: print('skip', config, seed); continue
        t0 = time.time()
        sig, pe = train_eval(config, seed, SEP, SPAT)
        np.save(OUT / f'nb41_pred{TAG}_{config}_s{seed}.npy', pe)
        row = dict(config=config, seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t0))
        pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
        print(f'{config} seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
RES = pd.read_csv(CSVP); print(RES.to_string(index=False))

  resume distill s0 from epoch 14


distill seed 0: sigma_eff 0.0467 (4547s)


distill seed 1: sigma_eff 0.0481 (4727s)


nodistill seed 0: sigma_eff 0.0482 (3791s)


nodistill seed 1: sigma_eff 0.0470 (4360s)


   config  seed  sigma_eff  elapsed
  distill     0     0.0467     4547
  distill     1     0.0481     4727
nodistill     0     0.0482     3791
nodistill     1     0.0470     4360


## Verdict
H6 holds if `distill` beats `nodistill` (the identical-data control) by >= 0.002 on the real min-bias test split; smaller differences are noise at this sample size.

In [10]:
print('anchors: nb32 W4 seeds 0.0459-0.0471 (ens 0.0450) | nb35 final 0.0440 | '
      'per-bin targets 0.06/0.045/0.035/0.032/0.030/0.030 | diff < 0.002 not significant')
for config in CONFIGS:
    sub = RES[RES.config == config]
    if len(sub): print(f'{config:10s}: {sub.sigma_eff.mean():.4f} +/- '
                       f'{(sub.sigma_eff.std() if len(sub) > 1 else 0):.4f} (n={len(sub)})')
te_e = Ets[rte]
for config in CONFIGS:
    ps = [np.load(OUT / f'nb41_pred{TAG}_{config}_s{s}.npy') for s in SEEDS
          if (OUT / f'nb41_pred{TAG}_{config}_s{s}.npy').exists()]
    if len(ps) >= 2:
        print(f'{config} seed-ensemble ({len(ps)} seeds): '
              f'{resolution(np.stack(ps).mean(0), te_e)["sigma_eff"]:.4f}')
best_cfg = RES.groupby('config').sigma_eff.mean().idxmin()
ps = [np.load(OUT / f'nb41_pred{TAG}_{best_cfg}_s{s}.npy') for s in SEEDS
      if (OUT / f'nb41_pred{TAG}_{best_cfg}_s{s}.npy').exists()]
if ps:
    pe = np.stack(ps).mean(0)
    edges = np.quantile(te_e, np.linspace(0, 1, 7))
    print(f'per-E-bin sigma_eff ({best_cfg}):')
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        if mm.sum() >= 20:
            print(f'  E {edges[i]:6.1f}-{edges[i+1]:6.1f} GeV: '
                  f'{resolution(pe[mm], te_e[mm])["sigma_eff"]:.4f}  (n={int(mm.sum())})')

anchors: nb32 W4 seeds 0.0459-0.0471 (ens 0.0450) | nb35 final 0.0440 | per-bin targets 0.06/0.045/0.035/0.032/0.030/0.030 | diff < 0.002 not significant
distill   : 0.0474 +/- 0.0010 (n=2)
nodistill : 0.0476 +/- 0.0008 (n=2)
distill seed-ensemble (2 seeds): 0.0464
nodistill seed-ensemble (2 seeds): 0.0461
per-E-bin sigma_eff (distill):
  E    2.2-  10.7 GeV: 0.0726  (n=1814)
  E   10.7-  17.4 GeV: 0.0532  (n=1814)
  E   17.4-  24.0 GeV: 0.0389  (n=1814)
  E   24.0-  34.1 GeV: 0.0388  (n=1814)
  E   34.1-  53.1 GeV: 0.0378  (n=1814)
  E   53.1- 100.0 GeV: 0.0397  (n=1814)
